# 1A pilot v6 — Do the findings survive other prompts, larger models and real unaccented text?

> **Tóm tắt (VI):** v6 lấp các khoảng trống còn lại trước khi viết bài. **Không chạy lại v4/v5**; dùng lại đúng dữ liệu, code prompt, cách chấm và cổng hợp lệ của v4 (kiểm tra tự động).
> - **A. Độ bền theo prompt + giảm thiểu bằng prompt:** 3 mẫu prompt mới cho 16 mô hình của v4, trên `clean` và `strip`: (1) diễn đạt lại bằng tiếng Việt, (2) hướng dẫn bằng tiếng Anh, (3) prompt v4 **thêm lời nhắc** "văn bản có thể viết không dấu". Câu hỏi: mức giảm và thứ hạng mô hình có đổi theo prompt không? Lời nhắc lấy lại được bao nhiêu so với khôi phục dấu tự động (v5)?
> - **B. Mô hình lớn hơn:** Qwen2.5-14B, Qwen3-14B, phi-4 (14B) chạy toàn bộ quy trình v4 (+ `restored`). Câu hỏi: xu hướng theo kích thước có tiếp tục trên 8B không?
> - **C. Văn bản vốn không dấu:** các bình luận ViHSD (tập test) mà **chính người viết** gõ không dấu (v4 đã loại chúng), chấm nguyên bản và sau khi khôi phục dấu. Câu hỏi: kết quả trên văn bản bỏ dấu nhân tạo có đúng với văn bản không dấu thật không?
>
> **GPU:** mô hình 14B cần GPU ≥ 40 GB (**A100**). Trên L4 notebook chạy phần A và C cho 16 mô hình nhỏ rồi báo thiếu mô hình 14B; đổi sang A100 và `Run all` lại thì chỉ chạy phần còn thiếu (kết quả lưu theo từng mô hình). Cần secret `HF_TOKEN` (🔑, **không dán token vào cell**) để có Llama như v4. Kết quả ở `MyDrive/llm-research/1A-pilot-v6/`; v4 và v5 chỉ được đọc.

| part | models | conditions | prompt |
|---|---|---|---|
| A | v4's 16 (+ the 14B ones) | `clean`, `strip` | `para` (Vietnamese paraphrase), `en` (English instructions and label names), `hint` (v4 + a note that the text may lack diacritics) |
| B | Qwen2.5-14B-Instruct, Qwen3-14B, phi-4 | v4's 7 + `restored` (v5's restored texts) | v4's, with v4's format probe, free-generation check and validity gate |
| C | all | `natural` (ViHSD test comments written without diacritics), `natural_restored` | v4's |

**Fixed in advance:** template validity = the pair's v4 gate (or B's gate) **and**, under the new template, median letter mass ≥ 0.5 and clean accuracy above chance by ≥ 10 points with its 95% CI above chance. Template comparisons use the pairs valid under every template. The hint's effect is compared with v5's automatic restoration on the same pairs.

## Step 0 · Setup (as v4; v4 and v5 results are read only)

In [ ]:
import sys, os, re, json, ast, math, random, unicodedata, difflib, warnings, shutil, time, gc
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install -U transformers accelerate
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/llm-research"
else:
    ROOT = os.environ.get("PILOT_ROOT", "outputs")
SMOKE = bool(os.environ.get("PILOT_SMOKE"))   # local test only: tiny samples, no validity gate
SRC4 = f"{ROOT}/1A-pilot-v4" + ("-smoke" if SMOKE else "")   # read only
SRC5 = f"{ROOT}/1A-pilot-v5" + ("-smoke" if SMOKE else "")   # read only
OUT = f"{ROOT}/1A-pilot-v6" + ("-smoke" if SMOKE else "")
for d in ["data", "big/preds", "big/probe", "big/gen", "templates", "natural", "figures", "tables", "checks"]:
    os.makedirs(f"{OUT}/{d}", exist_ok=True)
LOCAL_MODELS = json.loads(os.environ.get("PILOT_LOCAL_MODELS", "{}"))  # local test only: {"hub name": "folder"}

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="An input array is constant")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)
print("v4:", SRC4, "| v5:", SRC5, "| v6 results:", OUT)

CHECKS = []   # every automatic check, saved to checks/checks.csv
def check(name, ok, detail=""):
    CHECKS.append({"check": name, "ok": bool(ok), "detail": str(detail)[:300]})
    print(("✅ " if ok else "❌ ") + name + (f" — {detail}" if detail else ""))
    pd.DataFrame(CHECKS).drop_duplicates("check", keep="last").to_csv(f"{OUT}/checks/checks.csv", index=False)
    return ok

def require(name, ok, detail=""):
    # a failed requirement stops the notebook: no numbers are better than wrong numbers
    if not check(name, ok, detail):
        raise RuntimeError(f"Stopped: {name}. {detail}")

In [ ]:
SEED = 42
TASKS = ["vsfc", "vsmec", "vihsd", "xnli", "vimmrc"]
CONDITIONS = ["clean", "partial25", "partial50", "partial75", "strip", "teencode", "mixed"]
NOISY = CONDITIONS[1:]
LEVEL = {"clean": 0, "partial25": 25, "partial50": 50, "partial75": 75, "strip": 100}

# items per class: test = the evaluation, dev = format choice and calibration (validation split, disjoint)
TEST_PER_CLASS = {"vsfc": 150, "vsmec": 35, "vihsd": 150, "xnli": 150, "vimmrc": 60}
DEV_PER_CLASS = {"vsfc": 30, "vsmec": 15, "vihsd": 30, "xnli": 30, "vimmrc": 20}
MIN_SYLLABLES = 3

MODEL_INFO = {  # hub name: (family, size in B, Vietnamese-adapted?)   small to large, so cheap ones finish first
    "Qwen/Qwen2.5-1.5B-Instruct": ("Qwen2.5", 1.5, False),
    "Qwen/Qwen3-1.7B": ("Qwen3", 1.7, False),
    "SeaLLMs/SeaLLMs-v3-1.5B-Chat": ("SeaLLMs-v3", 1.5, True),
    "bigscience/bloomz-1b7": ("bloomz", 1.7, False),
    "vilm/vinallama-2.7b-chat": ("VinaLLaMA", 2.7, True),
    "Qwen/Qwen2.5-3B-Instruct": ("Qwen2.5", 3.1, False),
    "bigscience/bloomz-3b": ("bloomz", 3.0, False),
    "Qwen/Qwen3-4B": ("Qwen3", 4.0, False),
    "microsoft/Phi-3.5-mini-instruct": ("Phi-3.5", 3.8, False),
    "Qwen/Qwen2.5-7B-Instruct": ("Qwen2.5", 7.6, False),
    "Qwen/Qwen3-8B": ("Qwen3", 8.2, False),
    "SeaLLMs/SeaLLMs-v3-7B-Chat": ("SeaLLMs-v3", 7.6, True),
    "vilm/vinallama-7b-chat": ("VinaLLaMA", 6.9, True),
    "bigscience/bloomz-7b1": ("bloomz", 7.1, False),
    "mistralai/Mistral-7B-Instruct-v0.3": ("Mistral", 7.2, False),
}
GATED_INFO = {"meta-llama/Llama-3.2-3B-Instruct": ("Llama-3", 3.2, False)}   # needs the HF_TOKEN secret

# Validity gate, fixed before any noisy result is seen (Step 5)
MIN_LETTER_MASS = 0.5     # median probability on the option letters, clean test
MIN_GEN_AGREEMENT = 0.7   # free greedy answer == scored answer
MIN_ABOVE_CHANCE = 0.10   # clean accuracy − chance, and its 95% CI lower bound must be > 0
B = 2000                  # bootstrap resamples
TOKEN_BUDGET = 4096       # padded tokens per forward batch (halved automatically on out-of-memory)
N_GEN = 40                # items per task × {clean, strip} for the free-generation check
if SMOKE:
    TEST_PER_CLASS = {t: 4 for t in TASKS}
    DEV_PER_CLASS = {t: 2 for t in TASKS}
    MIN_LETTER_MASS, MIN_GEN_AGREEMENT, MIN_ABOVE_CHANCE, B, N_GEN = 0.0, 0.0, -1.0, 200, 4

In [ ]:
# v6 settings
BIG_INFO = {"Qwen/Qwen2.5-14B-Instruct": ("Qwen2.5", 14.7, False),
            "Qwen/Qwen3-14B": ("Qwen3", 14.8, False),
            "microsoft/phi-4": ("Phi-4", 14.7, False)}
BIG = list(BIG_INFO)
MODEL_INFO.update(BIG_INFO)
BIG_MIN_GB = 34            # bf16 14B weights are ~30 GB
CONDITIONS_B = CONDITIONS + ["restored"]
TEMPLATES = ["v4", "para", "en", "hint"]
NEW_TEMPLATES = TEMPLATES[1:]
TPL_CONDS = ["clean", "strip"]
NAT_CONDS = ["natural", "natural_restored"]
RESTORER = "peterhung/vietnamese-accent-marker-xlm-roberta"   # v5's restorer
WINDOW = 120
MIN_RESTORED_SHARE = 0.2   # natural items: the restorer must add diacritics to ≥ 20% of words (else: not Vietnamese)
N_REPRO = 4 if SMOKE else 16
require("v4 results exist", os.path.exists(f"{SRC4}/checks/validity.csv"), SRC4)
require("v5 results exist", os.path.exists(f"{SRC5}/data/restored.csv"), SRC5)

In [ ]:
# Figure style: one colour per model family, the same in every figure
FAMILIES = ["Qwen2.5", "Qwen3", "SeaLLMs-v3", "VinaLLaMA", "bloomz", "Phi-3.5", "Mistral", "Llama-3"]
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
FAM_COLOR = dict(zip(FAMILIES, PALETTE))
COND_COLOR = {"partial25": "#cde2fb", "partial50": "#86b6ef", "partial75": "#3987e5", "strip": "#0d366b",
              "teencode": "#eb6834", "mixed": "#6250d6"}
TASK_NAME = {"vsfc": "VSFC (sentiment)", "vsmec": "VSMEC (emotion)", "vihsd": "ViHSD (hate speech)",
             "xnli": "XNLI-vi (inference)", "vimmrc": "ViMMRC (reading)"}
SEQ = mpl.colors.LinearSegmentedColormap.from_list("blue", ["#f4f8fd", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 220, "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#8a8984", "axes.labelcolor": "#2b2a28",
    "xtick.color": "#52514e", "ytick.color": "#52514e", "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6,
    "legend.frameon": False, "lines.linewidth": 2, "lines.markersize": 5, "pdf.fonttype": 42,
})
FIGS = []
def save(fig, name, caption):
    fig.savefig(f"{OUT}/figures/{name}.png", bbox_inches="tight")
    fig.savefig(f"{OUT}/figures/{name}.pdf", bbox_inches="tight")
    FIGS.append((name, caption))
    plt.show()

def save_table(df, name, caption, floatfmt="%.2f"):
    df.to_csv(f"{OUT}/tables/{name}.csv")
    try:
        df.to_latex(f"{OUT}/tables/{name}.tex", float_format=floatfmt, caption=caption, label=f"tab:{name}", na_rep="--")
    except Exception as e:
        print("latex export failed:", name, repr(e)[:100])

KEY = {}   # headline numbers → key_numbers.json and the conclusions

In [ ]:
FAM_COLOR["Phi-4"] = "#52514e"

## Step 1 · Data (v4's code), v4's noise, v5's restored texts — checked against v4 and v5

In [ ]:
PARQUET = "https://huggingface.co/datasets/{}/resolve/refs%2Fconvert%2Fparquet/{}/{}/0000.parquet"
LETTERS = "ABCDEFG"

TASK_SPECS = {
    "vsfc": dict(source=("uitnlp/vietnamese_students_feedback", "default"), label_col="sentiment", classes=[0, 1, 2],
                 labels=["tiêu cực", "trung tính", "tích cực"],
                 fields=[("sentence", "Nhận xét: ")],
                 intro="Đọc nhận xét sau của một sinh viên về môn học và cho biết thái độ của nhận xét.",
                 labels_en=["negative", "neutral", "positive"], fields_en=["Feedback: "],
                 intro_en="Read the following student feedback about a course and state its attitude."),
    "vsmec": dict(source=("ura-hcmut/UIT-VSMEC", "default"), label_col="Emotion",
                  classes=["Enjoyment", "Sadness", "Anger", "Fear", "Disgust", "Surprise", "Other"],
                  labels=["vui vẻ", "buồn bã", "tức giận", "sợ hãi", "ghê tởm", "ngạc nhiên", "khác"],
                  fields=[("Sentence", "Bình luận: ")],
                  intro="Đọc bình luận mạng xã hội sau và cho biết cảm xúc chính mà nó thể hiện.",
                  labels_en=["enjoyment", "sadness", "anger", "fear", "disgust", "surprise", "other"], fields_en=["Comment: "],
                  intro_en="Read the following social media comment and state the main emotion it expresses."),
    "vihsd": dict(source=("ura-hcmut/UIT-ViHSD", "default"), label_col="label_id", classes=[0, 1, 2],
                  labels=["bình thường (không có lời lẽ xấu)", "xúc phạm (lời lẽ thô tục nhưng không nhắm vào ai)",
                          "thù ghét (công kích, lăng mạ một người hay một nhóm người)"],
                  fields=[("free_text", "Bình luận: ")],
                  intro="Đọc bình luận mạng xã hội sau và phân loại nó.",
                  labels_en=["clean (no bad language)", "offensive (vulgar but not aimed at anyone)",
                             "hate (attacks or insults a person or a group)"], fields_en=["Comment: "],
                  intro_en="Read the following social media comment and classify it."),
    "xnli": dict(source=("facebook/xnli", "vi"), label_col="label", classes=[0, 1, 2],
                 labels=["kéo theo (giả thuyết chắc chắn đúng)", "trung lập (không xác định được)",
                         "mâu thuẫn (giả thuyết chắc chắn sai)"],
                 fields=[("premise", "Tiền đề: "), ("hypothesis", "Giả thuyết: ")],
                 intro="Cho một tiền đề và một giả thuyết. Hãy cho biết quan hệ giữa chúng.",
                 labels_en=["entailment (the hypothesis is definitely true)", "neutral (cannot be determined)",
                            "contradiction (the hypothesis is definitely false)"], fields_en=["Premise: ", "Hypothesis: "],
                 intro_en="Given a premise and a hypothesis, state the relation between them."),
    "vimmrc": dict(source=("ura-hcmut/ViMMRC", "default"), label_col="answer", classes=["A", "B", "C", "D"],
                   labels=None,   # the options are part of the item
                   fields=[("article", "Đoạn văn: "), ("question", "Câu hỏi: "), ("options", "Các lựa chọn:\n")],
                   intro="Đọc đoạn văn rồi trả lời câu hỏi trắc nghiệm.",
                   labels_en=None, fields_en=["Passage: ", "Question: ", "Options:\n"],
                   intro_en="Read the passage, then answer the multiple-choice question."),
}
K = {t: len(TASK_SPECS[t]["classes"]) for t in TASKS}
CHANCE = {t: 1 / K[t] for t in TASKS}

def nfc(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s))).strip()

def strip_diacritics(s):
    s = s.replace("đ", "d").replace("Đ", "D")
    s = unicodedata.normalize("NFD", s)
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return unicodedata.normalize("NFC", s)

def syllable_spans(text):
    return [(m.start(), m.end()) for m in re.finditer(r"\S+", text) if any(ch.isalpha() for ch in m.group())]

def accent_share(text):
    w = [x for x in text.split() if any(c.isalpha() for c in x)]
    return sum(strip_diacritics(x) != x for x in w) / max(len(w), 1)

def load_split(task, split):
    spec = TASK_SPECS[task]
    raw = pd.read_parquet(PARQUET.format(*spec["source"], split))
    if task == "vimmrc":
        # options as one field, one option per line; the letters are part of the text the model reads
        raw["options"] = ["\n".join(f"{l}. {nfc(o)}" for l, o in zip("ABCD", ast.literal_eval(x) if isinstance(x, str) else x))
                          for x in raw.options]
    fields = [f for f, _ in spec["fields"]]
    for f in fields:
        raw[f] = raw[f].map(nfc) if f != "options" else raw[f].map(lambda s: "\n".join(nfc(x) for x in s.split("\n")))
    raw = raw[raw[spec["label_col"]].isin(spec["classes"])].copy()
    raw["key"] = raw[fields].agg("\n".join, axis=1)
    n0 = len(raw)
    dup = raw.key.map(raw.key.value_counts()) > 1
    noisy_src = raw[[f for f in fields if f != "options"]].agg(" ".join, axis=1)
    short = noisy_src.map(lambda s: len(syllable_spans(s))) < MIN_SYLLABLES
    no_diac = raw.key.map(strip_diacritics) == raw.key
    keep = raw[~dup & ~short & ~no_diac].copy()
    log = {"task": task, "split": split, "rows": n0, "duplicates": int(dup.sum()), "short": int(short.sum()),
           "no_diacritics": int(no_diac.sum()), "kept": len(keep)}
    return keep, log

def sample(df, task, per_class, seed):
    spec = TASK_SPECS[task]
    n_min = df[spec["label_col"]].value_counts().min()
    require(f"{task}: enough items per class", n_min >= per_class, f"smallest class {n_min} < {per_class}")
    s = df.groupby(spec["label_col"]).sample(n=per_class, random_state=seed).sample(frac=1, random_state=seed)
    s = s.reset_index(drop=True)
    s["label"] = s[spec["label_col"]].map(spec["classes"].index)
    s["item_id"] = range(len(s))
    return s

RAW, DEV, TEST, clean_log = {}, {}, {}, []
for t in TASKS:
    test, lt = load_split(t, "test")
    dev, ld = load_split(t, "validation")
    before = len(dev)
    dev = dev[~dev.key.isin(set(test.key))]
    ld["overlap_with_test"] = before - len(dev)
    clean_log += [lt, ld]
    RAW[t] = test
    TEST[t] = sample(test, t, TEST_PER_CLASS[t], SEED)
    DEV[t] = sample(dev, t, DEV_PER_CLASS[t], SEED + 1)
clean_log = pd.DataFrame(clean_log).set_index(["task", "split"])
save_table(clean_log, "table_data_cleaning", "Rows removed before sampling.", "%d")
display(clean_log)
for t in TASKS:
    require(f"{t}: dev and test share no text", not set(DEV[t].key) & set(TEST[t].key))
    require(f"{t}: test sample is balanced", TEST[t].label.value_counts().nunique() == 1)

In [ ]:
def partial_strip(text, rate, rng):
    words = text.split(" ")
    cand = [i for i, w in enumerate(words) if strip_diacritics(w) != w]
    chosen = set(rng.sample(cand, round(rate * len(cand))))
    return " ".join(strip_diacritics(w) if i in chosen else w for i, w in enumerate(words))

TEENCODE = {
    "giáo viên": "gv", "giảng viên": "gv", "sinh viên": "sv", "bài tập": "bt", "học sinh": "hs",
    "không": "ko", "được": "dc", "gì": "j", "với": "vs", "mình": "mk", "rồi": "r",
    "vậy": "v", "bạn": "bn", "quá": "wa", "nhiều": "nhìu", "người": "ng", "trước": "trc",
    "cũng": "cx", "thôi": "thui", "biết": "bít", "yêu": "iu", "giờ": "h",
    "nhưng": "nhg", "lắm": "lém", "thích": "thik", "những": "nhữg", "hiểu": "hỉu",
}
_teen_re = re.compile(r"(?<!\w)(" + "|".join(sorted(map(re.escape, TEENCODE), key=len, reverse=True)) + r")(?!\w)",
                      re.IGNORECASE)

def teencode(text):
    return _teen_re.sub(lambda m: TEENCODE[m.group(0).lower()], text)

def make_noisy(text, cond, item_key, field):
    rng = random.Random(f"{SEED}|{item_key}|{cond}|{field}")
    if cond == "clean":
        return text
    if cond == "strip":
        return strip_diacritics(text)
    if cond.startswith("partial"):
        return partial_strip(text, int(cond[len("partial"):]) / 100, rng)
    if cond == "teencode":
        return teencode(text)
    if cond == "mixed":
        return partial_strip(teencode(text), 0.5, rng)
    raise ValueError(cond)

def expand(df, task, conds):
    fields = [f for f, _ in TASK_SPECS[task]["fields"]]
    rows = []
    for r in df.itertuples(index=False):
        r = r._asdict()
        for c in conds:
            row = {"task": task, "item_id": r["item_id"], "label": r["label"], "cond": c}
            for f in fields:
                noisy = r[f] if f == "options" and c == "clean" else make_noisy(r[f], c, r["key"], f)
                row[f], row[f + "_clean"] = noisy, r[f]
            row["text"] = "\n".join(row[f] for f in fields)
            row["clean"] = "\n".join(row[f + "_clean"] for f in fields)
            rows.append(row)
    return pd.DataFrame(rows)

DATA = {t: expand(TEST[t], t, CONDITIONS) for t in TASKS}
DEVDATA = {t: expand(DEV[t], t, ["clean"]) for t in TASKS}
pd.concat(DATA.values())[["task", "item_id", "label", "cond", "text", "clean"]].to_csv(f"{OUT}/data/test_items.csv", index=False)
pd.concat(DEVDATA.values())[["task", "item_id", "label", "text"]].to_csv(f"{OUT}/data/dev_items.csv", index=False)

again = expand(TEST["vsfc"], "vsfc", CONDITIONS)
require("noise is deterministic", again.text.equals(DATA["vsfc"].text))
assert strip_diacritics("Tôi muốn đặt vé máy bay") == "Toi muon dat ve may bay"

rows = []
for t in TASKS:
    d = DATA[t]
    for c in NOISY:
        g = d[d.cond == c]
        row = {"task": t, "cond": c, "items changed": (g.text != g.clean).mean(),
               "syllables changed": np.mean([1 - difflib.SequenceMatcher(None, a.split(), b.split(), autojunk=False).ratio()
                                             for a, b in zip(g.clean, g.text)])}
        if c in LEVEL:
            row["only diacritics changed"] = (g.text.map(strip_diacritics) == g.clean.map(strip_diacritics)).mean()
        rows.append(row)
noise_tab = pd.DataFrame(rows)
noise_tab.to_csv(f"{OUT}/checks/noise.csv", index=False)
st = noise_tab[noise_tab.cond == "strip"]
require("strip changes every test item", (st["items changed"] == 1).all(), st.set_index("task")["items changed"].round(3).to_dict())
require("strip/partial change only diacritics", (noise_tab.dropna()["only diacritics changed"] == 1).all())
require("strip leaves no diacritic", all((DATA[t][DATA[t].cond == "strip"].text.map(strip_diacritics) ==
                                          DATA[t][DATA[t].cond == "strip"].text).all() for t in TASKS))
hit = tot = 0
for t in TASKS:
    g = DATA[t][DATA[t].cond == "partial50"]
    for r in g.itertuples():
        for f, _ in TASK_SPECS[t]["fields"]:
            a, b = getattr(r, f + "_clean").split(" "), getattr(r, f).split(" ")
            cand = [i for i, w in enumerate(a) if strip_diacritics(w) != w]
            hit += sum(a[i] != b[i] for i in cand)
            tot += len(cand)
check("partial50 strips ~50% of accented words", abs(hit / tot - 0.5) < 0.02, f"realized {hit / tot:.3f}")
display(noise_tab.pivot(index="task", columns="cond", values="items changed")[NOISY].round(2))
ex = DATA["vsfc"][DATA["vsfc"].item_id == 0].set_index("cond").text.reindex(CONDITIONS).to_frame("example (VSFC)")
save_table(ex, "table_noise_example", "One VSFC test item under every condition (original Vietnamese text).")
display(ex)

In [ ]:
old = pd.read_csv(f"{SRC4}/data/test_items.csv", keep_default_na=False)
new = pd.concat(DATA.values())[["task", "item_id", "label", "cond", "text", "clean"]].reset_index(drop=True)
require("test items, labels and noisy texts identical to v4", len(old) == len(new) and (old.astype(str).values == new.astype(str).values).all(),
        f"{len(old)} vs {len(new)} rows")

def fields_of(t):
    return [f for f, _ in TASK_SPECS[t]["fields"]]

cached = pd.read_csv(f"{SRC5}/data/restored.csv", keep_default_na=False)
units = [(t, r["item_id"], f, r[f]) for t in TASKS for r in DATA[t][DATA[t].cond == "strip"].to_dict("records") for f in fields_of(t)]
require("v5's restored texts belong to these strip texts", len(cached) == len(units) and
        (cached.stripped.values == np.array([u[3] for u in units])).all())
RMAP = {(r.task, r.item_id, r.field): r.restored for r in cached.itertuples()}
for t in TASKS:
    s = DATA[t][DATA[t].cond == "strip"].copy()
    s["cond"] = "restored"
    for f in fields_of(t):
        s[f] = [RMAP[(t, i, f)] for i in s.item_id]
    s["text"] = ["\n".join(r[f] for f in fields_of(t)) for r in s.to_dict("records")]
    DATA[t] = pd.concat([DATA[t], s], ignore_index=True)
v5i = pd.read_csv(f"{SRC5}/data/test_items.csv", keep_default_na=False)
mine = pd.concat(DATA.values())
mine = mine[mine.cond == "restored"][["task", "item_id", "text"]].reset_index(drop=True)
theirs = v5i[v5i.cond == "restored"][["task", "item_id", "text"]].reset_index(drop=True)
require("restored texts identical to v5", len(mine) == len(theirs) and (mine.astype(str).values == theirs.astype(str).values).all())
pd.concat(DATA.values())[["task", "item_id", "label", "cond", "text", "clean"]].to_csv(f"{OUT}/data/test_items.csv", index=False)

### Part C data: ViHSD comments their authors wrote without diacritics

v4 removed test items without any diacritic (they cannot be stripped). Here they are the test set: same cleaning as v4 (NFC, valid label, no duplicates, ≥ 3 syllables), then restored with v5's restorer. Comments where the restorer adds a diacritic to fewer than 20% of the words are dropped: they are mostly English, names or slang, not unaccented Vietnamese. All remaining comments are used (not class-balanced: the source is mostly `clean`), so accuracy is reported with **balanced accuracy** and macro-F1 next to it.

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from huggingface_hub import hf_hub_download
import torch
RCACHE = f"{OUT}/data/restored.csv"
WINDOW = 120

def fields_of(t):
    return [f for f, _ in TASK_SPECS[t]["fields"]]

def recase(src, low):
    # put the capitalisation of src back on its lower-cased rewrite (same length)
    return "".join(c.upper() if s.isupper() else c for s, c in zip(src, low)) if len(src) == len(low) else low

def restore_texts(texts):
    rtok = AutoTokenizer.from_pretrained(LOCAL_MODELS.get(RESTORER, RESTORER), add_prefix_space=True)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    rmodel = AutoModelForTokenClassification.from_pretrained(LOCAL_MODELS.get(RESTORER, RESTORER)).to(dev).eval()
    tag_file = (os.path.join(LOCAL_MODELS[RESTORER], "selected_tags_names.txt") if RESTORER in LOCAL_MODELS
                else hf_hub_download(RESTORER, "selected_tags_names.txt"))
    tags = [l.strip() for l in open(tag_file, encoding="utf-8") if l.strip()]
    require("restorer: tag set as on the model card", len(tags) == rmodel.config.num_labels, f"{len(tags)} tags")
    tags = [tuple(x.split("-")) for x in tags]
    # windows of words: (text index, line index, start word) → words
    lines = [t.split("\n") for t in texts]
    jobs = []
    for i, ls in enumerate(lines):
        for j, line in enumerate(ls):
            words = line.split(" ")
            for s in range(0, len(words), WINDOW):
                jobs.append((i, j, s, words[s:s + WINDOW]))
    out_words = {}
    for b in range(0, len(jobs), 32):
        batch = jobs[b:b + 32]
        keep = [[w for w in ws if w] for _, _, _, ws in batch]
        enc = rtok([k if k else ["."] for k in keep], is_split_into_words=True, truncation=True, max_length=512,
                   padding=True, return_tensors="pt")
        with torch.no_grad():
            pred = rmodel(**{k: v.to(dev) for k, v in enc.items()}).logits.argmax(-1).cpu().numpy()
        for n, (i, j, s, ws) in enumerate(batch):
            labs = {}
            for p, w in enumerate(enc.word_ids(n)):
                if w is not None:
                    labs.setdefault(w, []).append(pred[n, p])
            res, w_i = [], 0
            for w in ws:
                if not w:
                    res.append(w)
                    continue
                low, new = w.lower(), None
                for li in labs.get(w_i, []):   # the first tag that changes the word, as on the model card
                    raw, vowel = tags[li]
                    if raw and raw in low:
                        new = low.replace(raw, vowel)
                        break
                res.append(recase(w, new) if new is not None else w)
                w_i += 1
            out_words[(i, j, s)] = res
    del rmodel
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    restored = []
    for i, ls in enumerate(lines):
        new_lines = []
        for j, line in enumerate(ls):
            words = line.split(" ")
            new_lines.append(" ".join(sum((out_words[(i, j, s)] for s in range(0, len(words), WINDOW)), [])))
        restored.append("\n".join(new_lines))
    return restored


RCACHE = f"{OUT}/data/natural_restored.csv"

def natural_items():
    spec = TASK_SPECS["vihsd"]
    raw = pd.read_parquet(PARQUET.format(*spec["source"], "test"))
    raw["free_text"] = raw.free_text.map(nfc)
    raw = raw[raw[spec["label_col"]].isin(spec["classes"])].copy()
    dup = raw.free_text.map(raw.free_text.value_counts()) > 1
    short = raw.free_text.map(lambda s: len(syllable_spans(s))) < MIN_SYLLABLES
    plain = raw.free_text.map(strip_diacritics) == raw.free_text
    nat = raw[~dup & ~short & plain].copy().reset_index(drop=True)
    nat["label"] = nat[spec["label_col"]].map(spec["classes"].index)
    return nat, {"test rows": len(raw), "no diacritics (after v4's cleaning)": len(nat)}

NAT, nat_log = natural_items()
require("natural items share no text with v4's test sample", not set(NAT.free_text) & set(TEST["vihsd"].free_text))
if SMOKE:
    NAT = NAT.groupby("label").head(4).reset_index(drop=True)
cachedn = pd.read_csv(RCACHE, keep_default_na=False) if os.path.exists(RCACHE) else None
if cachedn is not None and len(cachedn) == len(NAT) and (cachedn.natural.values == NAT.free_text.values).all():
    print("restored texts read from", RCACHE)
else:
    cachedn = pd.DataFrame({"natural": NAT.free_text, "restored": restore_texts(list(NAT.free_text))})
    cachedn.to_csv(RCACHE, index=False)
NAT["restored"] = cachedn.restored.values
require("natural restoration changes only diacritics", (NAT.restored.map(strip_diacritics) == NAT.free_text).all())
def changed_share(a, b):
    w = [(x, y) for x, y in zip(a.split(" "), b.split(" ")) if any(c.isalpha() for c in x)]
    return sum(x != y for x, y in w) / max(len(w), 1)
NAT["restored_share"] = [changed_share(a, b) for a, b in zip(NAT.free_text, NAT.restored)]
nat_log["restorer adds diacritics to < 20% of words (dropped)"] = int((NAT.restored_share < MIN_RESTORED_SHARE).sum())
if not SMOKE:
    NAT = NAT[NAT.restored_share >= MIN_RESTORED_SHARE].reset_index(drop=True)
NAT["item_id"] = range(len(NAT))
nat_log["used"] = len(NAT)
nat_log.update({f"class {TASK_SPECS['vihsd']['labels_en'][k].split(' ')[0]}": int((NAT.label == k).sum()) for k in range(K["vihsd"])})
NATDATA = pd.concat([NAT.assign(cond="natural", text=NAT.free_text),
                     NAT.assign(cond="natural_restored", text=NAT.restored, free_text=NAT.restored)], ignore_index=True)
NATDATA["task"] = "vihsd"
NATDATA[["task", "item_id", "label", "cond", "text", "restored_share"]].to_csv(f"{OUT}/data/natural_items.csv", index=False)
save_table(pd.Series(nat_log).to_frame("count"), "table_natural_items", "ViHSD test comments written without diacritics: selection.", "%d")
display(pd.Series(nat_log).to_frame("count"))
check("enough natural items (≥ 150)", SMOKE or len(NAT) >= 150, len(NAT))
display(NAT.sample(min(6, len(NAT)), random_state=SEED)[["free_text", "restored", "label", "restored_share"]])

## Step 3 · Models and prompts: look before running (CPU)

For every model, using its tokenizer only:
1. **Prompt format**: the model's own chat template; VinaLLaMA-2.7b ships without one, so it gets the ChatML template from VinaLLaMA-7b (both model cards document ChatML); bloomz is not a chat model and gets a plain prompt. The exact prompt for one item is printed.
2. **Answer letters**: each option letter must be **one token** right after the prompt, and the letters must be different tokens. Checked for every prefill candidate.
3. **Length**: the longest prompt must fit the model's context.
4. **Tokenizer statistics** of the input fields *inside the prompt* (what the model actually reads): tokens per syllable, share of syllables split into several tokens.

A model that fails a check is listed and left out; nothing runs on a broken setup.

In [ ]:
HF_TOKEN = None
if IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
MODEL_INFO.update(GATED_INFO if (HF_TOKEN or any(m in LOCAL_MODELS for m in GATED_INFO)) else {})
print("gated models:", "included" if any(m in MODEL_INFO for m in GATED_INFO) else "skipped (no HF_TOKEN secret)")
SHORT = {m: m.split("/")[1].replace("-Instruct", "").replace("-Chat", "").replace("-chat", "").replace("-mini-instruct", "-mini")
         .replace("-v0.3", "") for m in MODEL_INFO}

from transformers import AutoTokenizer
CHAT_TEMPLATE_FROM = {"vilm/vinallama-2.7b-chat": "vilm/vinallama-7b-chat"}   # documented ChatML, missing in the 2.7b tokenizer
PREFILLS = {"chat": ["Đáp án:", "", "**"], "plain": ["Đáp án:", ""]}

def load_tokenizer(name):
    tok = AutoTokenizer.from_pretrained(LOCAL_MODELS.get(name, name), token=HF_TOKEN)
    source = "own chat template" if tok.chat_template else "plain prompt (not a chat model)"
    if name in CHAT_TEMPLATE_FROM:
        donor = CHAT_TEMPLATE_FROM[name]
        tok.chat_template = AutoTokenizer.from_pretrained(LOCAL_MODELS.get(donor, donor), token=HF_TOKEN).chat_template
        source = f"ChatML from {donor.split('/')[1]}"
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    return tok, source

def build_message(task, row):
    # returns the user message and the character span of every input field inside it
    spec = TASK_SPECS[task]
    parts = [(spec["intro"] + "\n\n", None)]
    for f, prefix in spec["fields"]:
        parts += [(prefix, None), (row[f], f), ("\n", None)]
    if spec["labels"]:
        parts.append(("\nCác lựa chọn:\n" + "\n".join(f"{L}. {o}" for L, o in zip(LETTERS, spec["labels"])) + "\n", None))
    letters = LETTERS[:K[task]]
    parts.append(("\nChỉ trả lời bằng một chữ cái: " + ", ".join(letters[:-1]) + " hoặc " + letters[-1] + ".", None))
    msg, spans = "", {}
    for text, f in parts:
        if f:
            spans[f] = (len(msg), len(msg) + len(text))
        msg += text
    return msg, spans

_WRAP = {}
def chat_wrapper(tok):
    # the chat template rendered once around a placeholder; the template copies the message verbatim (checked)
    if id(tok) not in _WRAP:
        ph = "<<<MESSAGE>>>"
        w = tok.apply_chat_template([{"role": "user", "content": ph}], tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False)
        assert w.count(ph) == 1, "chat template does not copy the message verbatim"
        test_msg = "Thử: tiếng Việt có dấu.\nDòng hai."
        full = tok.apply_chat_template([{"role": "user", "content": test_msg}], tokenize=False, add_generation_prompt=True,
                                       enable_thinking=False)
        assert full == w.replace(ph, test_msg), "chat template changes the message"
        _WRAP[id(tok)] = w.split(ph)
    return _WRAP[id(tok)]

def build_prompt(tok, task, row, prefill):
    msg, spans = build_message(task, row)
    if tok.chat_template:
        head, tail = chat_wrapper(tok)
        off = len(head)
        prompt, add_special = head + msg + tail + prefill, False
    else:
        off, prompt, add_special = 0, msg + "\n" + prefill, True
    return prompt, add_special, {f: (a + off, b + off) for f, (a, b) in spans.items()}

def encode(tok, task, row, prefill):
    # token ids, the token positions of each field, and per-field tokenizer statistics
    prompt, add_special, spans = build_prompt(tok, task, row, prefill)
    enc = tok(prompt, add_special_tokens=add_special, return_offsets_mapping=True)
    ids, offs = enc["input_ids"], enc["offset_mapping"]
    pos, stats = {}, {}
    for f, (a, b) in spans.items():
        p = [i for i, (x, y) in enumerate(offs) if y > x and x < b and y > a and i > 0]
        sy = [(a + s, a + e) for s, e in syllable_spans(prompt[a:b])]
        # a syllable is "broken" if more than one token covers it; offsets are sorted, so walk them once
        tok_spans = [offs[i] for i in p]
        j, broken = 0, 0
        for s_, e_ in sy:
            while j < len(tok_spans) and tok_spans[j][1] <= s_:
                j += 1
            k, n = j, 0
            while k < len(tok_spans) and tok_spans[k][0] < e_:
                n += 1
                k += 1
            broken += n > 1
        pos[f] = p
        stats[f] = (len(p), len(sy), broken)
    return prompt, add_special, ids, pos, stats

_LETTERS = {}
def letter_ids(tok, prompt, add_special, ids, task):
    # for each option letter: token ids of " X" / "X" that are ONE token right after the prompt.
    # Every prompt of a task ends with the same text, so the answer is cached on that ending (and re-checked on its tokens).
    key = (id(tok), task, prompt[-80:], tuple(ids[-6:]))
    if key in _LETTERS:
        return _LETTERS[key]
    out = []
    for L in LETTERS[:K[task]]:
        v = []
        for s in (" " + L, L):
            full = tok(prompt + s, add_special_tokens=add_special)["input_ids"]
            if full[:len(ids)] == ids and len(full) == len(ids) + 1:
                v.append(full[-1])
        out.append(sorted(set(v)))
    _LETTERS[key] = out
    return out

def context_len(name, tok):
    from transformers import AutoConfig
    cfg = AutoConfig.from_pretrained(LOCAL_MODELS.get(name, name), token=HF_TOKEN)
    n = getattr(cfg, "max_position_embeddings", None) or getattr(cfg, "seq_length", None) or getattr(cfg, "n_positions", None)
    return n or 2048   # bloom uses ALiBi (no hard limit); 2048 is its training length

In [ ]:
# Part A: the prompt templates. "v4" must give v4's message exactly (checked below).
_build_message_v4 = build_message
TPL = {"name": "v4"}
PARA_INTRO = {
    "vsfc": "Dưới đây là ý kiến của một sinh viên về một môn học. Ý kiến này mang thái độ nào?",
    "vsmec": "Dưới đây là một bình luận trên mạng xã hội. Người viết chủ yếu thể hiện cảm xúc nào?",
    "vihsd": "Dưới đây là một bình luận trên mạng xã hội. Bình luận này thuộc loại nào?",
    "xnli": "Dưới đây là một tiền đề và một giả thuyết. Giả thuyết có quan hệ thế nào với tiền đề?",
    "vimmrc": "Hãy đọc kỹ đoạn văn dưới đây rồi chọn câu trả lời đúng cho câu hỏi.",
}
HINT = ("Lưu ý: văn bản dưới đây có thể được viết không dấu hoặc thiếu dấu. "
        "Hãy hiểu nó như tiếng Việt có dấu bình thường.")
PREFILL_MAP = {"v4": {}, "para": {}, "hint": {}, "en": {"Đáp án:": "Answer:"}}

def build_message(task, row):
    tpl = TPL["name"]
    if tpl == "v4":
        return _build_message_v4(task, row)
    spec = TASK_SPECS[task]
    letters = LETTERS[:K[task]]
    prefixes = [p for _, p in spec["fields"]]
    labels, head = spec["labels"], "\nCác lựa chọn:\n"
    ask = "\nChỉ trả lời bằng một chữ cái: " + ", ".join(letters[:-1]) + " hoặc " + letters[-1] + "."
    intro = spec["intro"]
    if tpl == "hint":
        intro = intro + "\n" + HINT
    elif tpl == "para":
        intro, head = PARA_INTRO[task], "\nPhương án:\n"
        prefixes = [p.replace("Các lựa chọn:", "Phương án:") for p in prefixes]
        ask = "\nTrả lời bằng đúng một chữ cái (" + "/".join(letters) + ")."
    elif tpl == "en":
        intro, prefixes, labels, head = spec["intro_en"], spec["fields_en"], spec["labels_en"], "\nOptions:\n"
        ask = "\nAnswer with one letter only: " + ", ".join(letters[:-1]) + " or " + letters[-1] + "."
    parts = [(intro + "\n\n", None)]
    for (f, _), prefix in zip(spec["fields"], prefixes):
        parts += [(prefix, None), (row[f], f), ("\n", None)]
    if labels:
        parts.append((head + "\n".join(f"{L}. {o}" for L, o in zip(LETTERS, labels)) + "\n", None))
    parts.append((ask, None))
    msg, spans = "", {}
    for text, f in parts:
        if f:
            spans[f] = (len(msg), len(msg) + len(text))
        msg += text
    return msg, spans

class template:
    # with template("en"): ... — always restores v4's prompt afterwards
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        TPL["name"] = self.name
    def __exit__(self, *a):
        TPL["name"] = "v4"

same = all(build_message(t, r) == _build_message_v4(t, r) for t in TASKS for r in DATA[t].head(20).to_dict("records"))
require("template 'v4' is exactly v4's prompt", same)
for tpl in NEW_TEMPLATES:
    with template(tpl):
        ok = all(build_message(t, r)[0][slice(*build_message(t, r)[1][f])] == r[f]
                 for t in TASKS for r in DATA[t].head(10).to_dict("records") for f in fields_of(t))
    require(f"template '{tpl}': field spans point at the input text", ok)
for tpl in TEMPLATES:
    with template(tpl):
        print(f"──── {tpl} (XNLI, strip)\n" + build_message("xnli", DATA["xnli"][DATA["xnli"].cond == "strip"].iloc[0])[0] + "\n")

In [ ]:
TOK, setup_rows, TOKSTATS = {}, [], []
for name in MODEL_INFO:
    try:
        tok, source = load_tokenizer(name)
    except Exception as e:
        check(f"{SHORT[name]}: tokenizer loads", False, repr(e)[:150])
        continue
    kind = "chat" if tok.chat_template else "plain"
    ok, notes, longest = True, [], 0
    for t in TASKS:
        rows = DATA[t].to_dict("records")
        for pre in PREFILLS[kind]:
            r = rows[0]
            prompt, add, ids, pos, _ = encode(tok, t, r, pre)
            lid = letter_ids(tok, prompt, add, ids, t)
            firsts = [v[0] for v in lid if v]
            if not all(lid) or len(set(firsts)) != len(firsts):
                notes.append(f"{t}/prefill {pre!r}: letters not single distinct tokens")
                if pre == "Đáp án:":
                    ok = False
        for r in [x for x in rows if x["cond"] in ("clean", "strip", "restored")]:   # tokenizer statistics (v6: the conditions it needs)
            _, _, ids, _, stats = encode(tok, t, r, "Đáp án:")
            longest = max(longest, len(ids))
            TOKSTATS.append({"model": name, "task": t, "item_id": r["item_id"], "cond": r["cond"],
                             **{f"{k}_{f}": v for f, s in stats.items() for k, v in zip(["ntok", "nsyl", "nbroken"], s)}})
    ctx = context_len(name, tok)
    ok &= longest + 16 <= ctx
    TOK[name] = tok
    setup_rows.append({"model": SHORT[name], "family": MODEL_INFO[name][0], "size_B": MODEL_INFO[name][1],
                       "vi_adapted": MODEL_INFO[name][2], "prompt format": source, "longest prompt": longest,
                       "context": ctx, "ok": ok, "notes": "; ".join(notes)})
    check(f"{SHORT[name]}: letters are single tokens and the longest prompt fits", ok, "; ".join(notes) or f"{longest}/{ctx} tokens")
setup = pd.DataFrame(setup_rows).set_index("model")
save_table(setup.drop(columns="notes"), "table_models", "Models, prompt format, and prompt length.")
display(setup)
MODELS = [m for m in MODEL_INFO if m in TOK and setup.loc[SHORT[m], "ok"]]
print(len(MODELS), "models pass the setup checks")
TOKSTATS = pd.DataFrame(TOKSTATS)
TOKSTATS.to_csv(f"{OUT}/data/tokenizer_stats.csv", index=False)

In [ ]:
v4_models = sorted(pd.read_csv(f"{SRC4}/checks/validity.csv").model.unique())
missing = [m for m in v4_models + BIG if m not in MODELS]
check("every v4 model and every 14B model is available", not missing,
      f"missing {missing}" + (" — add the HF_TOKEN secret (🔑) for Llama and Run all again" if any("llama" in m for m in missing) else ""))

# the option letters must be single, distinct tokens under every template (with the prefill each template uses)
rows = []
for name in MODELS:
    tok = TOK[name]
    kind = "chat" if tok.chat_template else "plain"
    for t in TASKS:
        r = DATA[t].iloc[0]
        for tpl in NEW_TEMPLATES:
            with template(tpl):
                for pre in PREFILLS[kind]:
                    pre = PREFILL_MAP[tpl].get(pre, pre)
                    lid = letter_ids(tok, *encode(tok, t, r, pre)[:3], t)
                    firsts = [v[0] for v in lid if v]
                    rows.append({"model": name, "task": t, "template": tpl, "prefill": pre,
                                 "letters_ok": bool(all(lid) and len(set(firsts)) == len(firsts))})
LETTER_OK = pd.DataFrame(rows)
LETTER_OK.to_csv(f"{OUT}/checks/template_letters.csv", index=False)
LOK = {(r.model, r.task, r.template, r.prefill): r.letters_ok for r in LETTER_OK.itertuples()}
check("option letters are single tokens under the new templates", LETTER_OK.letters_ok.mean() > 0.95,
      f"{LETTER_OK.letters_ok.mean():.0%} of (model, task, template, prefill)")

## Step 2 · Run (GPU)

Per model (each loaded once):
- **B** (14B models only): v4's `run_task` unchanged — format probe on dev, calibration prior, all conditions + `restored`, free-generation check;
- **A**: for each new template, the calibration prior on v4's dev items with that template, then `clean` and `strip`; for v4's models, 16 clean items re-scored with v4's prompt must reproduce v4's answers;
- **C**: `natural` and `natural_restored` with v4's prompt, prefill and a prior recomputed on v4's dev items.

The 14B models are skipped when the GPU has less than 34 GB; Run all again on an A100 to add them.

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from huggingface_hub import constants as hf_constants
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = (torch.bfloat16 if DEVICE == "cuda" and torch.cuda.get_device_capability()[0] >= 8
         else torch.float16 if DEVICE == "cuda" else torch.float32)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "", "| dtype:", DTYPE)

def fname(name):
    return name.replace("/", "__")

def pad_batch(seqs, pad_id):
    Lm = max(map(len, seqs))
    ids = torch.full((len(seqs), Lm), pad_id)
    mask = torch.zeros((len(seqs), Lm), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s)
        mask[i, :len(s)] = 1
    return ids.to(DEVICE), mask.to(DEVICE)

@torch.no_grad()
def forward_batch(model, tok, items):
    # items: dicts with ids, letters (list of token-id lists), pos (field → token positions)
    ids, mask = pad_batch([it["ids"] for it in items], tok.pad_token_id)
    logits = model(input_ids=ids, attention_mask=mask).logits
    last = logits[torch.arange(len(items), device=DEVICE), mask.sum(1) - 1].float().log_softmax(-1)
    out = []
    # gather log p(token_t | tokens < t) for every field token, in chunks to keep memory bounded
    pairs = [(b, p) for b, it in enumerate(items) for f in it["pos"] for p in it["pos"][f]]
    tok_lp = {}
    for i in range(0, len(pairs), 512):
        chunk = pairs[i:i + 512]
        bb = torch.tensor([b for b, _ in chunk], device=DEVICE)
        pp = torch.tensor([p for _, p in chunk], device=DEVICE)
        sel = logits[bb, pp - 1].float()
        lp = sel.gather(1, ids[bb, pp][:, None]).squeeze(1) - sel.logsumexp(-1)
        tok_lp.update(zip(chunk, lp.tolist()))
    del logits
    for b, it in enumerate(items):
        letter_lp = [torch.logsumexp(last[b, v], 0).item() for v in it["letters"]]
        nll = {f: -sum(tok_lp[(b, p)] for p in it["pos"][f]) for f in it["pos"]}
        out.append((letter_lp, nll))
    return out

def batches(items, budget):
    order = sorted(range(len(items)), key=lambda i: -len(items[i]["ids"]))
    cur, longest = [], 0
    for i in order:
        n = len(items[i]["ids"])
        if cur and max(longest, n) * (len(cur) + 1) > budget:
            yield cur
            cur, longest = [], 0
        cur.append(i)
        longest = max(longest, n)
    if cur:
        yield cur

def score(model, tok, task, rows, prefill, budget):
    items = []
    for r in rows:
        prompt, add, ids, pos, stats = encode(tok, task, r, prefill)
        items.append({"ids": ids, "pos": pos, "letters": letter_ids(tok, prompt, add, ids, task)})
    res = [None] * len(items)
    for idx in batches(items, budget):
        for i, o in zip(idx, forward_batch(model, tok, [items[i] for i in idx])):
            res[i] = o
    lp = np.array([r[0] for r in res])
    nll = pd.DataFrame([r[1] for r in res]).add_prefix("nll_")
    return lp, nll

def softmax_rows(lp):
    p = np.exp(lp - lp.max(1, keepdims=True))
    return p / p.sum(1, keepdims=True)

@torch.no_grad()
def generate(model, tok, task, rows, prefill, max_new_tokens=12):
    tok.padding_side = "left"
    out = []
    for i in range(0, len(rows), 8):
        enc = [encode(tok, task, r, prefill) for r in rows[i:i + 8]]
        Lm = max(len(e[2]) for e in enc)
        ids = torch.tensor([[tok.pad_token_id] * (Lm - len(e[2])) + e[2] for e in enc], device=DEVICE)
        mask = (torch.arange(Lm)[None, :] >= torch.tensor([Lm - len(e[2]) for e in enc])[:, None]).long().to(DEVICE)
        g = model.generate(input_ids=ids, attention_mask=mask, max_new_tokens=max_new_tokens, do_sample=False,
                           pad_token_id=tok.pad_token_id)
        out += [tok.decode(x[Lm:], skip_special_tokens=True) for x in g]
    tok.padding_side = "right"
    return out

def parse_letter(text, task):
    m = re.search(r"\b([A-G])\b", text.replace("*", " ")[:40])
    return LETTERS.index(m.group(1)) if m and LETTERS.index(m.group(1)) < K[task] else -1

In [ ]:
def run_task(model, tok, name, task, budget):
    kind = "chat" if tok.chat_template else "plain"
    # 1. format probe on dev (label-blind)
    dev_rows = DEVDATA[task].to_dict("records")
    probe = []
    for pre in PREFILLS[kind]:
        lid = letter_ids(tok, *encode(tok, task, dev_rows[0], pre)[:3], task)
        if not all(lid):
            continue
        lp, _ = score(model, tok, task, dev_rows, pre, budget)
        mass = np.exp(lp).sum(1)
        probe.append({"model": name, "task": task, "prefill": pre, "median_mass": float(np.median(mass)),
                      "dev_acc_raw": float((lp.argmax(1) == DEVDATA[task].label.values).mean()), "lp": lp})
    best = max(probe, key=lambda p: p["median_mass"])
    pre = best["prefill"]
    # 2. calibration prior from the clean dev items, with the chosen prefill
    prior = softmax_rows(best["lp"]).mean(0)
    # 3. test run
    rows = DATA[task].to_dict("records")
    lp, nll = score(model, tok, task, rows, pre, budget)
    p = softmax_rows(lp)
    out = DATA[task][["task", "item_id", "label", "cond"]].reset_index(drop=True)
    out["model"] = name
    for k in range(K[task]):
        out[f"lp{k}"] = lp[:, k]
    out["letter_mass"] = np.exp(lp).sum(1)
    out["pred_raw"] = p.argmax(1)
    out["pred"] = (np.log(p) - np.log(prior)).argmax(1)
    out = pd.concat([out, nll], axis=1)
    out["prefill"] = pre
    # 4. free-generation check
    sub = DATA[task][DATA[task].item_id.isin(DATA[task].item_id.unique()[:N_GEN]) & DATA[task].cond.isin(["clean", "strip"])]
    gen_rows = sub.to_dict("records")
    answers = generate(model, tok, task, gen_rows, pre)
    gen = sub[["task", "item_id", "label", "cond"]].assign(model=name, answer=answers,
                                                           parsed=[parse_letter(a, task) for a in answers])
    gen = gen.merge(out[["item_id", "cond", "pred_raw"]], on=["item_id", "cond"])
    probe_df = pd.DataFrame([{k: v for k, v in pr.items() if k != "lp"} for pr in probe]).assign(
        chosen=lambda d: d.prefill == pre, prior=str(np.round(prior, 3).tolist()))
    return out, probe_df, gen


def big_done(name, t):
    return all(os.path.exists(f"{OUT}/big/{d}/{t}__{fname(name)}.csv") for d in ["preds", "probe", "gen"])

def tpl_done(name, t):
    return os.path.exists(f"{OUT}/templates/{t}__{fname(name)}.csv")

def nat_done(name):
    return os.path.exists(f"{OUT}/natural/vihsd__{fname(name)}.csv")

V4VALID = pd.read_csv(f"{SRC4}/checks/validity.csv", keep_default_na=False)
PREFILL4 = {(r.model, r.task): r.prefill for r in V4VALID.itertuples()}

def prefill_of(name, t):
    if name in BIG:
        pr = pd.read_csv(f"{OUT}/big/probe/{t}__{fname(name)}.csv", keep_default_na=False)
        return pr[pr.chosen.astype(str) == "True"].prefill.iloc[0]
    return PREFILL4[(name, t)]

def scored(model, tok, name, task, rows, pre, prior, budget):
    lp, _ = score(model, tok, task, rows.to_dict("records"), pre, budget)
    p = softmax_rows(lp)
    out = rows[["task", "item_id", "label", "cond"]].reset_index(drop=True)
    out["model"] = name
    for k in range(K[task]):
        out[f"lp{k}"] = lp[:, k]
    out["letter_mass"] = np.exp(lp).sum(1)
    out["pred_raw"] = p.argmax(1)
    out["pred"] = (np.log(p) - np.log(prior)).argmax(1)
    out["prefill"] = pre
    out["prior"] = str(np.round(prior, 3).tolist())
    return out

def dev_prior(model, tok, task, pre, budget):
    lp, _ = score(model, tok, task, DEVDATA[task].to_dict("records"), pre, budget)
    return softmax_rows(lp).mean(0), float((lp.argmax(1) == DEVDATA[task].label.values).mean())

def run_templates(model, tok, name, task, budget):
    pre4 = prefill_of(name, task)
    rows = DATA[task][DATA[task].cond.isin(TPL_CONDS)]
    outs, repro = [], {}
    if name not in BIG:   # v4's prompt still gives v4's answers
        rep = DATA[task][DATA[task].cond == "clean"].head(N_REPRO)
        lp, _ = score(model, tok, task, rep.to_dict("records"), pre4, budget)
        ref = pd.read_csv(f"{SRC4}/preds/{task}__{fname(name)}.csv", keep_default_na=False)
        ref = ref[ref.cond == "clean"].set_index("item_id").loc[rep.item_id]
        repro = {"model": name, "task": task, "answers_match": float((lp.argmax(1) == ref.pred_raw.values).mean())}
    for tpl in NEW_TEMPLATES:
        pre = PREFILL_MAP[tpl].get(pre4, pre4)
        if not LOK.get((name, task, tpl, pre), False):
            print(f"   {task}/{tpl}: letters are not single tokens with prefill {pre!r}, skipped")
            continue
        with template(tpl):
            prior, dev_acc = dev_prior(model, tok, task, pre, budget)
            outs.append(scored(model, tok, name, task, rows, pre, prior, budget).assign(template=tpl, dev_acc_raw=dev_acc))
    return (pd.concat(outs, ignore_index=True) if outs else None), repro

def run_natural(model, tok, name, budget):
    pre = prefill_of(name, "vihsd")
    prior, _ = dev_prior(model, tok, "vihsd", pre, budget)
    return scored(model, tok, name, "vihsd", NATDATA, pre, prior, budget)

def with_budget(fn, *args):
    budget = TOKEN_BUDGET
    while True:
        try:
            return fn(*args, budget)
        except torch.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache()
            budget //= 2
            print(f"   out of memory, token budget → {budget}")
            if budget < 512:
                return None

In [ ]:
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if DEVICE == "cuda" else 0
BIG_FITS = SMOKE or GPU_GB >= BIG_MIN_GB
RUN_MODELS = [m for m in MODELS if m in v4_models] + [m for m in BIG if m in MODELS]
def todo_of(name):
    return {"big": [t for t in TASKS if name in BIG and not big_done(name, t)],
            "tpl": [t for t in TASKS if not tpl_done(name, t)], "nat": not nat_done(name)}
pending = {m: todo_of(m) for m in RUN_MODELS}
pending = {m: d for m, d in pending.items() if d["big"] or d["tpl"] or d["nat"]}
print(f"GPU {GPU_GB:.0f} GB — 14B models {'run' if BIG_FITS else 'skipped (need an A100)'}; {len(pending)} models with work left")
if pending and DEVICE != "cuda" and not SMOKE:
    raise RuntimeError("Model runs are missing and there is no GPU: Runtime → Change runtime type → A100 (or L4 for the small models)")
REPRO_FILE = f"{OUT}/checks/reproduction.csv"
REPRO = pd.read_csv(REPRO_FILE).to_dict("records") if os.path.exists(REPRO_FILE) else []
failed, skipped_big = {}, []
for name, todo in pending.items():
    if name in BIG and not BIG_FITS:
        skipped_big.append(name)
        continue
    t0 = time.time()
    try:
        tok = TOK[name]
        model = AutoModelForCausalLM.from_pretrained(LOCAL_MODELS.get(name, name), dtype=DTYPE, token=HF_TOKEN).to(DEVICE).eval()
    except Exception as e:
        failed[name] = repr(e)[:200]
        print("✗ could not load", name, failed[name])
        continue
    print(f"● {SHORT[name]} loaded in {time.time() - t0:.0f}s")
    for t in todo["big"]:
        t1 = time.time()
        res = with_budget(lambda b: run_task(model, tok, name, t, b))
        if res is None:
            failed[f"{name}/{t}/big"] = "out of memory"
            continue
        out, probe_df, gen = res
        out.to_csv(f"{OUT}/big/preds/{t}__{fname(name)}.csv", index=False)
        probe_df.to_csv(f"{OUT}/big/probe/{t}__{fname(name)}.csv", index=False)
        gen.to_csv(f"{OUT}/big/gen/{t}__{fname(name)}.csv", index=False)
        acc = (out.pred == out.label).groupby(out.cond).mean()
        print(f"   B {t:7} {time.time() - t1:4.0f}s  clean={acc['clean']:.2f} strip={acc['strip']:.2f} restored={acc['restored']:.2f}"
              f"  gen-agree={(gen.parsed == gen.pred_raw).mean():.2f}")
    for t in todo["tpl"]:
        if name in BIG and not big_done(name, t):
            continue
        t1 = time.time()
        res = with_budget(lambda b: run_templates(model, tok, name, t, b))
        if res is None or res[0] is None:
            failed[f"{name}/{t}/templates"] = "out of memory" if res is None else "no template could be scored"
            continue
        out, repro = res
        out.to_csv(f"{OUT}/templates/{t}__{fname(name)}.csv", index=False)
        if repro:
            REPRO = [r for r in REPRO if (r["model"], r["task"]) != (name, t)] + [repro]
            pd.DataFrame(REPRO).to_csv(REPRO_FILE, index=False)
        acc = (out.pred == out.label).groupby([out.template, out.cond]).mean().unstack()
        print(f"   A {t:7} {time.time() - t1:4.0f}s  " + "  ".join(f"{k}: {r['clean']:.2f}→{r['strip']:.2f}" for k, r in acc.iterrows())
              + (f"  | v4 answers reproduced {repro['answers_match']:.0%}" if repro else ""))
    if todo["nat"] and (name not in BIG or big_done(name, "vihsd")):
        out = with_budget(lambda b: run_natural(model, tok, name, b))
        if out is None:
            failed[f"{name}/natural"] = "out of memory"
        else:
            out.to_csv(f"{OUT}/natural/vihsd__{fname(name)}.csv", index=False)
            acc = (out.pred == out.label).groupby(out.cond).mean()
            print(f"   C vihsd   natural={acc['natural']:.2f} restored={acc['natural_restored']:.2f}")
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    if IN_COLAB:   # 14B weights are ~30 GB; keep Colab's disk free
        shutil.rmtree(os.path.join(hf_constants.HF_HUB_CACHE, "models--" + name.replace("/", "--")), ignore_errors=True)
small_done = all(tpl_done(m, t) and nat_done(m) for m in RUN_MODELS if m not in BIG for t in TASKS)
big_all = all(big_done(m, t) and tpl_done(m, t) and nat_done(m) for m in BIG if m in RUN_MODELS for t in TASKS)
check("every run of v4's models finished", small_done and not {k: v for k, v in failed.items() if k.split("/")[0] not in BIG}, failed or "")
check("every run of the 14B models finished", big_all,
      "skipped on this GPU — Runtime → Change runtime type → A100, then Run all" if skipped_big else (failed or ""))

## Step 3 · Audit, validity gates, shared bootstrap

From here on, CPU only; the analysis uses whatever has finished (the conclusions say which models are in).

In [ ]:
import glob
from sklearn.metrics import f1_score
def rd(p):
    return pd.read_csv(p, keep_default_na=False)
SMALL = [m for m in RUN_MODELS if m not in BIG]
BIGDONE = [m for m in BIG if m in RUN_MODELS and all(big_done(m, t) for t in TASKS)]
MALL = SMALL + BIGDONE
FAMILY = {m: MODEL_INFO[m][0] for m in MALL}
SIZE = {m: MODEL_INFO[m][1] for m in MALL}

# v4-template predictions: v4 (+ v5 restored) for v4's models, B for the 14B models
base = [rd(f"{SRC4}/preds/{t}__{fname(m)}.csv") for m in SMALL for t in TASKS]
base += [rd(f"{SRC5}/preds/{t}__{fname(m)}.csv").query("cond == 'restored'") for m in SMALL for t in TASKS
         if os.path.exists(f"{SRC5}/preds/{t}__{fname(m)}.csv")]
base += [rd(f"{OUT}/big/preds/{t}__{fname(m)}.csv") for m in BIGDONE for t in TASKS]
BASE = pd.concat(base, ignore_index=True)
BASE["correct"] = (BASE.pred == BASE.label).astype(int)
TP = pd.concat([rd(p) for m in MALL for t in TASKS for p in [f"{OUT}/templates/{t}__{fname(m)}.csv"] if os.path.exists(p)],
               ignore_index=True)
TP["correct"] = (TP.pred == TP.label).astype(int)
NP = pd.concat([rd(p) for m in MALL for p in [f"{OUT}/natural/vihsd__{fname(m)}.csv"] if os.path.exists(p)], ignore_index=True)
NP["correct"] = (NP.pred == NP.label).astype(int)

rows = []
for (m, t, tpl), g in TP.groupby(["model", "task", "template"]):
    lp = g[[f"lp{k}" for k in range(K[t])]].to_numpy(float)
    rows.append({"model": SHORT[m], "task": t, "template": tpl, "rows": len(g), "expected": len(TPL_CONDS) * DATA[t].item_id.nunique(),
                 "duplicates": int(g.duplicated(["item_id", "cond"]).sum()), "finite": bool(np.isfinite(lp).all())})
for m, g in NP.groupby("model"):
    rows.append({"model": SHORT[m], "task": "vihsd", "template": "natural", "rows": len(g), "expected": len(NATDATA),
                 "duplicates": int(g.duplicated(["item_id", "cond"]).sum()), "finite": bool(np.isfinite(g.filter(regex="^lp").to_numpy(float)).all())})
integ = pd.DataFrame(rows)
integ.to_csv(f"{OUT}/checks/integrity.csv", index=False)
bad = integ[(integ.rows != integ.expected) | (integ.duplicates > 0) | ~integ.finite]
require("all new prediction files complete and consistent", bad.empty, bad.to_dict("records")[:3])
if os.path.exists(REPRO_FILE):
    REP = pd.read_csv(REPRO_FILE)
    check("v4's answers reproduced with the v4 prompt (≥ 95%)", SMOKE or REP.answers_match.mean() >= 0.95,
          f"mean {REP.answers_match.mean():.3f}, worst {REP.answers_match.min():.2f}")

rng = np.random.default_rng(SEED)
N_ITEMS = {t: DATA[t].item_id.nunique() for t in TASKS}
IDX = {t: rng.integers(0, N_ITEMS[t], (B, N_ITEMS[t])) for t in TASKS}   # the same resamples as v4

# v4's gate for v4's models; the same gate, computed here, for the 14B models
VB = []
for m in BIGDONE:
    for t in TASKS:
        g = BASE[(BASE.model == m) & (BASE.task == t)]
        gen = rd(f"{OUT}/big/gen/{t}__{fname(m)}.csv")
        c = g[g.cond == "clean"].sort_values("item_id")
        boot = c.correct.to_numpy(float)[IDX[t]].mean(1)
        r = {"model": m, "task": t, "prefill": g.prefill.iloc[0], "letter_mass": c.letter_mass.median(),
             "gen_agree": (gen.parsed == gen.pred_raw).mean(), "acc_clean": c.correct.mean(),
             "acc_lo": np.percentile(boot, 2.5), "acc_hi": np.percentile(boot, 97.5)}
        reasons = []
        if r["letter_mass"] < MIN_LETTER_MASS: reasons.append("letter mass")
        if r["gen_agree"] < MIN_GEN_AGREEMENT: reasons.append("generation disagrees")
        if r["acc_clean"] - CHANCE[t] < MIN_ABOVE_CHANCE or r["acc_lo"] <= CHANCE[t]: reasons.append("near chance")
        r["valid"], r["excluded_because"] = not reasons, ", ".join(reasons)
        VB.append(r)
VB = pd.DataFrame(VB, columns=["model", "task", "prefill", "letter_mass", "gen_agree", "acc_clean", "acc_lo", "acc_hi", "valid", "excluded_because"])
VB.to_csv(f"{OUT}/checks/validity_big.csv", index=False)
OK = {(r.model, r.task) for r in V4VALID.itertuples() if str(r.valid) == "True" and r.model in SMALL}
OK |= {(r.model, r.task) for r in VB.itertuples() if r.valid}
print(f"{len(MALL)} models ({len(BIGDONE)} of 14B); valid (model, task) pairs: {len(OK)}")
if len(VB):
    display(VB.assign(model=VB.model.map(SHORT)).set_index(["model", "task"]).round(2))

def vec(df, m, t, cond):
    return df[(df.model == m) & (df.task == t) & (df.cond == cond)].sort_values("item_id").correct.to_numpy(float)

def ci(a):
    a = np.asarray(a, float)
    a = a[np.isfinite(a)]
    return (np.percentile(a, 2.5), np.percentile(a, 97.5)) if len(a) else (np.nan, np.nan)

def fmt_ci(v, lo, hi):
    return f"{v:.2f} [95% CI {lo:.2f}, {hi:.2f}]"

## Step 4 · Part A — Do the results depend on the prompt?

**ACC[(model, task, template)]** = per-item correctness under `clean` and `strip`. A pair is valid under a template if it is valid in v4 (or B) and, under that template, median letter mass ≥ 0.5 and clean accuracy is ≥ 10 points above chance with its 95% CI above chance.

- **Normalised drop** per template (as in v4), mean over the pairs valid under **all four** templates, 95% paired bootstrap CI over items, and its difference from the v4 template.
- **Rank stability:** Spearman correlation between models' normalised drops under v4's prompt and under each other template (per task, and pooled over tasks).

In [ ]:
from scipy.stats import spearmanr
ACCT = {}
for m in MALL:
    for t in TASKS:
        ACCT[(m, t, "v4")] = np.stack([vec(BASE, m, t, c) for c in TPL_CONDS], 1)
        for tpl in NEW_TEMPLATES:
            g = TP[(TP.model == m) & (TP.task == t) & (TP.template == tpl)]
            if len(g):
                ACCT[(m, t, tpl)] = np.stack([vec(g, m, t, c) for c in TPL_CONDS], 1)

def mass_of(m, t, tpl):
    df = BASE if tpl == "v4" else TP[TP.template == tpl]
    return df[(df.model == m) & (df.task == t) & (df.cond == "clean")].letter_mass.median()

rows = []
for (m, t, tpl), a in ACCT.items():
    if not len(a):
        continue
    boot = a[IDX[t], 0].mean(1)
    acc = a[:, 0].mean()
    ok = (m, t) in OK and (tpl == "v4" or (mass_of(m, t, tpl) >= MIN_LETTER_MASS and acc - CHANCE[t] >= MIN_ABOVE_CHANCE
                                            and np.percentile(boot, 2.5) > CHANCE[t]))
    rows.append({"model": m, "task": t, "template": tpl, "acc_clean": acc, "acc_strip": a[:, 1].mean(),
                 "letter_mass": mass_of(m, t, tpl), "nd": (acc - a[:, 1].mean()) / (acc - CHANCE[t]), "valid": bool(ok)})
TV = pd.DataFrame(rows)
TV.to_csv(f"{OUT}/checks/validity_templates.csv", index=False)
display(TV.groupby(["template"]).valid.agg(["sum", "count"]).rename(columns={"sum": "valid pairs", "count": "pairs"}))
VALIDT = {(r.model, r.task, r.template) for r in TV.itertuples() if r.valid}
COMMON = {t: [m for m in MALL if all((m, t, tpl) in VALIDT for tpl in TEMPLATES)] for t in TASKS}

def nd_boot(a, t):
    bs = a[IDX[t]].mean(1)
    den = bs[:, 0] - CHANCE[t]
    return (bs[:, 0] - bs[:, 1]) / np.where(den > 0.05, den, np.nan)

rows = []
for t in TASKS:
    ms = COMMON[t]
    if not ms:
        continue
    bv4 = np.nanmean(np.stack([nd_boot(ACCT[(m, t, "v4")], t) for m in ms]), 0)
    for tpl in TEMPLATES:
        b = np.nanmean(np.stack([nd_boot(ACCT[(m, t, tpl)], t) for m in ms]), 0)
        pt = TV[(TV.task == t) & (TV.template == tpl) & TV.model.isin(ms)]
        r = {"task": t, "template": tpl, "models": len(ms), "acc_clean": pt.acc_clean.mean(), "acc_strip": pt.acc_strip.mean(),
             "nd_strip": pt.nd.mean()}
        r["nd_lo"], r["nd_hi"] = ci(b)
        d = b - bv4
        r["diff_vs_v4"] = pt.nd.mean() - TV[(TV.task == t) & (TV.template == "v4") & TV.model.isin(ms)].nd.mean()
        r["diff_lo"], r["diff_hi"] = ci(d)
        if tpl != "v4" and len(ms) >= 4:
            x = TV[(TV.task == t) & (TV.template == "v4")].set_index("model").nd.loc[ms]
            y = TV[(TV.task == t) & (TV.template == tpl)].set_index("model").nd.loc[ms]
            r["spearman_vs_v4"] = spearmanr(x, y).correlation
        rows.append(r)
TPLTAB = pd.DataFrame(rows).set_index(["task", "template"])
save_table(TPLTAB, "table_templates", "Normalised drop under strip for four prompt templates, on the (model, task) pairs valid under all four "
           "(95% CI over items); difference from v4's prompt; Spearman correlation of the models' drops with v4's prompt.")
display(TPLTAB.round(2))

# pooled over tasks: each model's mean normalised drop over the tasks where it is in COMMON
pool = TV[[(r.model in COMMON[r.task]) for r in TV.itertuples()]].pivot_table(index="model", columns="template", values="nd")
RANK = {tpl: spearmanr(pool["v4"], pool[tpl]).correlation for tpl in NEW_TEMPLATES} if len(pool) >= 4 else {}
pool_ci = {}
for tpl in TEMPLATES:   # mean over all common (model, task), bootstrap per task
    b = np.concatenate([np.stack([nd_boot(ACCT[(m, t, tpl)], t) for m in COMMON[t]]) for t in TASKS if COMMON[t]])
    pool_ci[tpl] = (np.nanmean([TV.set_index(["model", "task", "template"]).nd.loc[(m, t, tpl)] for t in TASKS for m in COMMON[t]]),
                    *ci(np.nanmean(b, 0)))
KEY["templates_pooled_nd"] = {k: list(v) for k, v in pool_ci.items()}
KEY["templates_rank_spearman_pooled"] = RANK
KEY["templates_common_pairs"] = sum(len(v) for v in COMMON.values())
print("pooled normalised drop (mean over common pairs):", {k: fmt_ci(*v) for k, v in pool_ci.items()})
print("rank correlation of models' pooled drops with v4's prompt:", {k: round(v, 2) for k, v in RANK.items()})

In [ ]:
TCOL = {"v4": "#0d366b", "para": "#3987e5", "en": "#86b6ef", "hint": "#eb6834"}
fig, ax = plt.subplots(figsize=(9, 3.6))
ts = [t for t in TASKS if COMMON[t]]
x = np.arange(len(ts))
for j, tpl in enumerate(TEMPLATES):
    g = TPLTAB.reset_index().set_index("task")
    g = g[g.template == tpl].loc[ts]
    ax.bar(x + (j - 1.5) * 0.2, g.nd_strip, 0.18, color=TCOL[tpl], label=tpl,
           yerr=[g.nd_strip - g.nd_lo, g.nd_hi - g.nd_strip], error_kw=dict(lw=1, ecolor="#52514e"))
ax.axhline(0, color="#8a8984", lw=1)
ax.set(xticks=x, xticklabels=[f"{TASK_NAME[t]}\n({len(COMMON[t])} models)" for t in ts], ylabel="normalised drop under strip",
       title="Loss without diacritics under four prompts (pairs valid under all four, 95% CI)")
ax.legend(fontsize=7, ncol=4)
plt.tight_layout()
save(fig, "fig13_templates", "Mean normalised drop under full stripping for v4's prompt, a Vietnamese paraphrase, English instructions, "
     "and v4's prompt with a note that the text may lack diacritics (pairs valid under all four templates, 95% CI).")

### Part A — Does telling the model about missing diacritics help, compared with restoring them?

On pairs valid under v4's prompt and the hint prompt, with `restored` available (v5 / B):
- **Δ clean**, **Δ strip** = accuracy with the hint − accuracy with v4's prompt;
- **recovery by the hint** = (strip with hint − strip with v4) ÷ (clean − strip, v4's prompt);
- **recovery by restoration** (v5's measure) on the same pairs.

In [ ]:
ACCR = {(m, t): vec(BASE, m, t, "restored") for m in MALL for t in TASKS}
rows, per = [], []
for t in TASKS:
    ms = [m for m in MALL if (m, t, "v4") in VALIDT and (m, t, "hint") in VALIDT and len(ACCR[(m, t)])]
    if not ms:
        continue
    bd = {k: [] for k in ["d_clean", "d_strip", "rec_hint", "rec_restore"]}
    for m in ms:
        a4, ah, ar = ACCT[(m, t, "v4")], ACCT[(m, t, "hint")], ACCR[(m, t)]
        lost = a4[:, 0].mean() - a4[:, 1].mean()
        per.append({"model": SHORT[m], "task": t, "d_clean": ah[:, 0].mean() - a4[:, 0].mean(), "d_strip": ah[:, 1].mean() - a4[:, 1].mean(),
                    "rec_hint": (ah[:, 1].mean() - a4[:, 1].mean()) / lost if lost > 0.05 else np.nan,
                    "rec_restore": (ar.mean() - a4[:, 1].mean()) / lost if lost > 0.05 else np.nan})
        b4, bh, br = a4[IDX[t]].mean(1), ah[IDX[t]].mean(1), ar[IDX[t]].mean(1)
        bl = b4[:, 0] - b4[:, 1]
        bl = np.where(bl > 0.05, bl, np.nan)
        bd["d_clean"].append(bh[:, 0] - b4[:, 0]); bd["d_strip"].append(bh[:, 1] - b4[:, 1])
        bd["rec_hint"].append((bh[:, 1] - b4[:, 1]) / bl); bd["rec_restore"].append((br - b4[:, 1]) / bl)
    pt = pd.DataFrame([p for p in per if p["task"] == t])
    r = {"task": t, "models": len(ms)}
    for k, v in bd.items():
        r[k] = pt[k].mean()
        r[k + "_lo"], r[k + "_hi"] = ci(np.nanmean(np.stack(v), 0))
    rows.append(r)
HINT_TAB = pd.DataFrame(rows).set_index("task")
HINT_PER = pd.DataFrame(per)
save_table(HINT_TAB, "table_hint_vs_restoration", "Prompt hint vs automatic restoration: accuracy change on clean and strip with the hint, "
           "and the share of the accuracy lost to stripping recovered by each (95% CI).")
save_table(HINT_PER.set_index(["model", "task"]), "table_hint_per_model", "Per valid (model, task): effect of the hint and of restoration.")
display(HINT_TAB.round(3))
KEY["hint"] = HINT_TAB.to_dict("index")

fig, ax = plt.subplots(figsize=(8, 3.4))
x = np.arange(len(HINT_TAB))
for j, (k, col, lab) in enumerate([("rec_hint", TCOL["hint"], "prompt hint"), ("rec_restore", "#1baf7a", "automatic restoration (v5)")]):
    ax.bar(x + (j - 0.5) * 0.36, HINT_TAB[k], 0.34, color=col, label=lab,
           yerr=[HINT_TAB[k] - HINT_TAB[k + "_lo"], HINT_TAB[k + "_hi"] - HINT_TAB[k]], error_kw=dict(lw=1, ecolor="#52514e"))
ax.axhline(0, color="#8a8984", lw=1)
ax.axhline(1, color="#8a8984", lw=1, ls=":")
ax.set(xticks=x, xticklabels=HINT_TAB.index, ylabel="share of lost accuracy recovered", title="Mitigation: telling the model vs restoring the text")
ax.legend(fontsize=7)
plt.tight_layout()
save(fig, "fig14_hint_vs_restoration", "Share of the accuracy lost to full stripping that is recovered by a prompt hint and by automatic "
     "diacritic restoration, on the same valid (model, task) pairs (95% CI).")

## Step 5 · Part B — Larger models

- the 14B models' normalised drop under every condition and their restoration recovery;
- **size trend:** the normalised drop under `strip` of every valid (model, task) regressed on log₂(size) with task and family fixed effects (standard errors clustered by model), with and without the 14B models; and within Qwen2.5 and Qwen3, which now span 1.5–15B;
- the tokenizer statistics of the 14B models (tokens per syllable, broken syllables), clean vs strip.

In [ ]:
import statsmodels.formula.api as smf
CONDS_B = CONDITIONS_B[1:]
rows = []
for m in BIGDONE:
    for t in TASKS:
        g = BASE[(BASE.model == m) & (BASE.task == t)]
        acc = g.groupby("cond").correct.mean()
        r = {"model": SHORT[m], "task": t, "valid": (m, t) in OK, "acc_clean": acc["clean"]}
        for c in CONDS_B:
            r[f"nd_{c}"] = (acc["clean"] - acc[c]) / (acc["clean"] - CHANCE[t])
        lost = acc["clean"] - acc["strip"]
        r["recovery"] = (acc["restored"] - acc["strip"]) / lost if lost > 0.05 else np.nan
        rows.append(r)
BIGTAB = pd.DataFrame(rows)
if len(BIGTAB):
    BIGTAB = BIGTAB.set_index(["model", "task"])
    save_table(BIGTAB, "table_big_models", "14B models: clean accuracy, normalised drop per condition, restoration recovery, validity.")
    display(BIGTAB.round(2))

nd = []
for m in MALL:
    for t in TASKS:
        if (m, t) not in OK:
            continue
        a = ACCT[(m, t, "v4")]
        nd.append({"model": m, "short": SHORT[m], "task": t, "family": FAMILY[m], "size": SIZE[m], "big": m in BIG,
                   "nd": (a[:, 0].mean() - a[:, 1].mean()) / (a[:, 0].mean() - CHANCE[t])})
ND = pd.DataFrame(nd)
ND["log2size"] = np.log2(ND["size"])
SIZEFIT = {}
for lab, d in [("≤ 8B (v4)", ND[~ND.big]), ("with 14B", ND)]:
    if d.model.nunique() < 4:
        continue
    f = smf.ols("nd ~ log2size + C(task) + C(family)", data=d).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d.model)[0]})
    SIZEFIT[lab] = {"slope_per_doubling": f.params["log2size"], "lo": f.conf_int().loc["log2size", 0], "hi": f.conf_int().loc["log2size", 1],
                    "p": f.pvalues["log2size"], "pairs": len(d), "models": d.model.nunique()}
SIZEFIT = pd.DataFrame(SIZEFIT).T
save_table(SIZEFIT, "table_size_trend", "Normalised drop under strip regressed on log2(model size), task and family fixed effects, "
           "SE clustered by model: change per doubling of size.", "%.4g")
display(SIZEFIT)
KEY["size_trend"] = SIZEFIT.to_dict("index")
fam_tab = ND[ND.family.isin(["Qwen2.5", "Qwen3"])].pivot_table(index=["family", "size"], columns="task", values="nd")
display(fam_tab.round(2))
save_table(fam_tab, "table_size_within_family", "Normalised drop under strip by size within Qwen2.5 and Qwen3 (valid pairs).")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
ax = axes[0]
mm = ND.groupby(["model", "family", "size"]).nd.mean().reset_index()
for fam, g in mm.groupby("family"):
    g = g.sort_values("size")
    ax.plot(g["size"], g.nd, marker="o", color=FAM_COLOR.get(fam, "#52514e"), lw=1.5 if len(g) > 1 else 0, label=fam)
ax.set(xscale="log", xlabel="parameters (B, log scale)", ylabel="mean normalised drop under strip",
       title="a. Loss vs size (mean over valid tasks)")
ax.set_xticks([1.5, 3, 7, 14]); ax.set_xticklabels(["1.5", "3", "7", "14"])
ax.xaxis.set_minor_formatter(mpl.ticker.NullFormatter())
ax.legend(fontsize=7, ncol=2)
ax = axes[1]
ts = TOKSTATS.copy()
ts["ntok"] = ts.filter(like="ntok_").sum(1); ts["nsyl"] = ts.filter(like="nsyl_").sum(1); ts["nbroken"] = ts.filter(like="nbroken_").sum(1)
tt = ts[ts.cond.isin(["clean", "strip"])].groupby(["model", "cond"])[["ntok", "nsyl", "nbroken"]].sum()
tt["tokens per syllable"] = tt.ntok / tt.nsyl
tt["broken share"] = tt.nbroken / tt.nsyl
tt = tt[["tokens per syllable", "broken share"]].unstack()
tt.index = tt.index.map(SHORT)
save_table(tt, "table_tokenizer_all", "Tokens per syllable and share of syllables split into several tokens, clean vs strip, every model.")
bs = tt["broken share"]
order = bs["clean"].sort_values().index
ax.scatter(bs.loc[order, "clean"], range(len(order)), color="#86b6ef", label="clean", zorder=3)
ax.scatter(bs.loc[order, "strip"], range(len(order)), color="#0d366b", label="strip", zorder=3)
for i, m in enumerate(order):
    ax.plot([bs.loc[m, "clean"], bs.loc[m, "strip"]], [i, i], color="#8a8984", lw=1, zorder=2)
ax.set(yticks=range(len(order)), yticklabels=order, xlabel="share of syllables split into several tokens",
       title="b. Broken syllables, clean vs strip")
ax.tick_params(axis="y", labelsize=7)
ax.legend(fontsize=7)
plt.tight_layout()
save(fig, "fig15_size_and_tokenizers", "a. Mean normalised drop under full stripping against model size, per family (valid pairs; "
     "the 14B models extend Qwen2.5 and Qwen3); b. share of syllables split into several tokens, clean vs strip, per model.")
display(tt.round(3))

## Step 6 · Part C — Text written without diacritics by its authors

For every model valid on ViHSD (v4's gate, or B's): balanced accuracy and macro-F1 on the comments as written (`natural`) and after automatic restoration (`natural_restored`), with a paired bootstrap CI over items for the difference. Next to it, the same models on v4's synthetic ViHSD items: stripped vs restored. If restoration helps on the natural comments too, the loss measured with synthetic stripping also happens on real unaccented text.

In [ ]:
def bal_acc(y, p):
    return np.mean([(p[y == k] == k).mean() for k in np.unique(y)])

nat_ms = [m for m in MALL if (m, "vihsd") in OK and m in set(NP.model)]
y = NAT.sort_values("item_id").label.to_numpy()
rng_n = np.random.default_rng(SEED + 7)
NIDX = rng_n.integers(0, len(y), (B, len(y)))
rows, boots = [], []
for m in nat_ms:
    g = NP[NP.model == m]
    pn = g[g.cond == "natural"].sort_values("item_id").pred.to_numpy()
    pr = g[g.cond == "natural_restored"].sort_values("item_id").pred.to_numpy()
    bn = np.array([bal_acc(y[i], pn[i]) for i in NIDX[:500]])
    br = np.array([bal_acc(y[i], pr[i]) for i in NIDX[:500]])
    s = BASE[(BASE.model == m) & (BASE.task == "vihsd")].groupby("cond").correct.mean()
    lost = s["clean"] - s["strip"]
    r = {"model": SHORT[m], "bal_acc_natural": bal_acc(y, pn), "bal_acc_restored": bal_acc(y, pr),
         "macroF1_natural": f1_score(y, pn, average="macro"), "macroF1_restored": f1_score(y, pr, average="macro"),
         "acc_natural": (pn == y).mean(), "acc_restored": (pr == y).mean(),
         "synthetic_clean": s["clean"], "synthetic_strip": s["strip"], "synthetic_restored": s["restored"],
         "synthetic_recovery": (s["restored"] - s["strip"]) / lost if lost > 0.05 else np.nan}
    r["gain"] = r["bal_acc_restored"] - r["bal_acc_natural"]
    r["gain_lo"], r["gain_hi"] = ci(br - bn)
    rows.append(r)
    boots.append(br - bn)
NATTAB = pd.DataFrame(rows).set_index("model") if rows else pd.DataFrame()
if len(NATTAB):
    mean_gain = np.mean(np.stack(boots), 0)
    KEY["natural"] = {"items": len(NAT), "models": len(nat_ms), "mean_gain_bal_acc": NATTAB.gain.mean(), "lo": ci(mean_gain)[0],
                      "hi": ci(mean_gain)[1], "models_gain_ci_above_0": int((NATTAB.gain_lo > 0).sum()),
                      "models_gain_ci_below_0": int((NATTAB.gain_hi < 0).sum()),
                      "synthetic_gain": float((NATTAB.synthetic_restored - NATTAB.synthetic_strip).mean()),
                      "restorer_changed_share": float(NAT.restored_share.mean())}
    save_table(NATTAB, "table_natural", "ViHSD comments written without diacritics: balanced accuracy and macro-F1 as written and after "
               "automatic restoration (gain with 95% CI), and the same models on v4's synthetic items.")
    display(NATTAB.round(3))
    print(KEY["natural"])

    fig, ax = plt.subplots(figsize=(7, 0.32 * len(NATTAB) + 1.4))
    d = NATTAB.sort_values("gain")
    yy = np.arange(len(d))
    ax.errorbar(d.gain, yy, xerr=[d.gain - d.gain_lo, d.gain_hi - d.gain], fmt="o", color="#0d366b", ms=5, lw=1.2, label="natural: restored − as written")
    ax.scatter(d.synthetic_restored - d.synthetic_strip, yy, marker="D", color="#1baf7a", s=22, zorder=3, label="synthetic: restored − strip")
    ax.axvline(0, color="#8a8984", lw=1)
    ax.set(yticks=yy, yticklabels=d.index, xlabel="gain from automatic restoration (balanced accuracy)",
           title=f"ViHSD comments written without diacritics (n = {len(NAT)})")
    ax.legend(fontsize=7, loc="lower right")
    plt.tight_layout()
    save(fig, "fig16_natural", "Gain in balanced accuracy from automatic diacritic restoration on ViHSD test comments their authors "
         "wrote without diacritics (95% CI over items), and the gain on v4's synthetically stripped ViHSD items.")

## Step 7 · Conclusions (auto-generated)

In [ ]:
L = ["# Pilot 1A v6 — prompts, larger models, natural unaccented text (auto-generated)", "",
     f"Models in this analysis: {len(SMALL)} from v4" + (f" + {', '.join(SHORT[m] for m in BIGDONE)}" if BIGDONE else
                                                          " (the 14B models have not run yet: run on an A100)") + ".", ""]
L.append("## A. Prompt templates")
pc = KEY["templates_pooled_nd"]
L.append(f"- On the {KEY['templates_common_pairs']} (model, task) pairs valid under all four prompts, the mean normalised drop under strip is "
         + "; ".join(f"{k} {fmt_ci(*v)}" for k, v in pc.items()) + ".")
if RANK:
    L.append("- Models' drops (mean over tasks) rank similarly across prompts: Spearman with v4's prompt "
             + ", ".join(f"{k} {v:.2f}" for k, v in RANK.items()) + ".")
big_diff = TPLTAB[(TPLTAB.diff_lo > 0) | (TPLTAB.diff_hi < 0)].reset_index()
big_diff = big_diff[big_diff.template != "v4"]
L.append("- Tasks where a template changes the drop (95% CI of the difference excludes 0): "
         + (", ".join(f"{r.task}/{r.template} ({r.diff_vs_v4:+.2f})" for r in big_diff.itertuples()) or "none") + ".")
L.append("")
L.append("## A. Prompt hint vs automatic restoration")
for t, r in HINT_TAB.iterrows():
    L.append(f"- **{TASK_NAME[t]}** ({int(r.models)} pairs): hint changes clean accuracy by {r.d_clean:+.3f} and strip accuracy by {r.d_strip:+.3f}; "
             f"it recovers {fmt_ci(r.rec_hint, r.rec_hint_lo, r.rec_hint_hi)} of the lost accuracy, restoration {fmt_ci(r.rec_restore, r.rec_restore_lo, r.rec_restore_hi)}.")
if len(HINT_TAB):
    L.append(f"- Mean over tasks: hint {HINT_TAB.rec_hint.mean():.0%}, restoration {HINT_TAB.rec_restore.mean():.0%}.")
L += ["", "## B. Larger models"]
if len(BIGTAB):
    for m, g in BIGTAB.reset_index().groupby("model"):
        v = g[g.valid]
        L.append(f"- **{m}**: valid on {len(v)}/{len(g)} tasks; mean normalised drop under strip {v.nd_strip.mean():.2f}, "
                 f"restoration recovers {v.recovery.mean():.0%}.")
for lab, r in SIZEFIT.iterrows():
    L.append(f"- Size trend ({lab}, {int(r.models)} models, {int(r.pairs)} pairs): {r.slope_per_doubling:+.3f} normalised drop per doubling "
             f"[95% CI {r.lo:+.3f}, {r.hi:+.3f}], p = {r.p:.3g}.")
L += ["", "## C. Natural unaccented text (ViHSD)"]
if "natural" in KEY:
    n = KEY["natural"]
    L.append(f"- {n['items']} test comments written without diacritics (the restorer changes {n['restorer_changed_share']:.0%} of their words). "
             f"Over {n['models']} valid models, restoration changes balanced accuracy by {fmt_ci(n['mean_gain_bal_acc'], n['lo'], n['hi'])} "
             f"(synthetic ViHSD: {n['synthetic_gain']:+.3f}); CI above 0 for {n['models_gain_ci_above_0']} models, below 0 for {n['models_gain_ci_below_0']}.")
L += ["", "## Notes for the paper",
      "- Template comparisons use only the pairs valid under every template; the free-generation check was not repeated for the new templates.",
      "- The natural set is not class-balanced (mostly `clean`); compare balanced accuracy, and compare natural with restored on the same items only.",
      "- The 14B models are bf16 like the others; their validity gate is v4's, computed here."]
TEXT = "\n".join(L)
open(f"{OUT}/conclusions.md", "w").write(TEXT)
display(Markdown(TEXT))

In [ ]:
def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [jsonable(v) for v in o]
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(f"{float(o):.4g}")   # 4 significant digits: keeps small p-values
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    return o
json.dump(jsonable(KEY), open(f"{OUT}/key_numbers.json", "w"), ensure_ascii=False, indent=1)
pd.DataFrame(FIGS, columns=["figure", "caption"]).to_csv(f"{OUT}/figures/captions.csv", index=False)
print("checks:", sum(c["ok"] for c in CHECKS), "passed,", sum(not c["ok"] for c in CHECKS), "failed")
print("figures:")
for n, c in FIGS:
    print(f"  {n}: {c}")
print("\nEverything is in", OUT, "— when done: Runtime → Disconnect and delete runtime.")